###Ingest circuits.csv file

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

#####Step 1 - Read the CSV file using the spark dataframe reader

In [0]:
 circuits_df = spark.read.option("header", True).csv("abfss://raw@dformulaone.dfs.core.windows.net/circuits.csv")

#####Step 2 - Set Schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [0]:
ciruits_schema = StructType(fields=[StructField("circuitId", IntegerType(), False),
                                    StructField("circuitRef", StringType(), True), 
                                    StructField("name", StringType(), True), 
                                    StructField("location", StringType(), True), 
                                    StructField("country", StringType(), True),
                                    StructField("lat", DoubleType(), True), 
                                    StructField("lng", DoubleType(), True), 
                                    StructField("alt", IntegerType(), True), 
                                    StructField("url", StringType(), True)])

In [0]:
 circuits_df = spark.read \
 .option("header", True) \
 .schema(ciruits_schema) \
 .csv(f"{raw_folder_path}/circuits.csv")

In [0]:
circuits_df.printSchema()

In [0]:
 circuits_df = spark.read \
 .option("header", True) \
 .schema(ciruits_schema) \
 .csv(f"{raw_folder_path}/circuits.csv")

In [0]:
from pyspark.sql.functions import col
circuits_selected_df =circuits_df.select(col("circuitId"), col("circuitRef"), col("name"), col("location"), col("country").alias("race_country"), col("lat"), col("lng"), col("alt"), col("url"))

#####Step 3 - Rename the columns as required 

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnRenamed("circuitId", "circuitid") \
    .withColumnRenamed("circuitRef", "circuit_ref") \
    .withColumnRenamed("lat", "latitude") \
    .withColumnRenamed("lng", "longitude") \
    .withColumnRenamed("alt", "altitude")

#####Step 4 - Add ingestion date to the dataframe

In [0]:
circuits_final_df = add_ingestion_date(circuits_renamed_df)

#####Step 5 - Write data to datalake as parquet

In [0]:
circuits_final_df.write.mode("overwrite").parquet(f"{processed_folder_path}/circuits")

In [0]:
%fs
ls abfss://processed@dformulaone.dfs.core.windows.net/circuits

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/circuits")
display(df, truncate=False)